# Download and clean the Heart Disease data

This notebook downloads the official UCI Cleveland heart-disease dataset and creates the project's binary-target CSV files. Run the code cell from top to bottom. It works whether Jupyter starts in the project root or in `notebooks/`.

## What the code does

1. Locates the project and creates the raw/processed data folders.
2. Downloads the original 14-column UCI file to `data/raw/heart.csv`.
3. Reads `?` as a missing value and displays counts, percentages, the exact source line/feature, and complete affected patient rows. Source lines are 1-based patient-record lines (the raw file has no header).
4. Drops a row only if its target is missing, maps target 0 to no disease and 1-4 to disease, and saves `data/processed/heart_clean.csv`.
5. Displays a second missing-value summary for the cleaned dataset. Missing predictors are intentionally retained here; the training pipeline imputes them using training data.

Running this cell downloads the source again and overwrites the two CSV files. The download requires internet access.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd

# Step 1: define the source and the 14 columns in the original UCI file.
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
COLUMNS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target_raw",
]

# Step 2: find the project folder whether Jupyter started here or in notebooks/.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "download.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project root containing download.py.")

raw_path = project_root / "data" / "raw" / "heart.csv"
clean_path = project_root / "data" / "processed" / "heart_clean.csv"
raw_path.parent.mkdir(parents=True, exist_ok=True)
clean_path.parent.mkdir(parents=True, exist_ok=True)

# Step 3: download a fresh raw copy. The UCI file uses '?' to mark missing values.
urlretrieve(URL, raw_path)
raw_data = pd.read_csv(
    raw_path,
    header=None,
    names=COLUMNS,
    na_values="?",
    skipinitialspace=True,
)
for column in COLUMNS:
    raw_data[column] = pd.to_numeric(raw_data[column], errors="coerce")

print("RAW DATA - missing-value audit")
print("In the original CSV, a question mark (?) means that a value was not recorded.")
print(f"Raw rows: {len(raw_data):,} | Raw columns: {len(COLUMNS)}")
missing_summary = pd.DataFrame({
    "missing_count": raw_data.isna().sum(),
    "missing_percent": (raw_data.isna().mean() * 100).round(2),
})
display(missing_summary)

# Show the exact source-file line and feature for every missing cell.
missing_cells = raw_data[COLUMNS].isna().stack()
missing_locations = (
    missing_cells[missing_cells]
    .rename_axis(index=["dataframe_index", "feature"])
    .reset_index(name="is_missing")
)
if missing_locations.empty:
    print("No missing cells found in the downloaded raw data.")
else:
    missing_locations["raw_file_line"] = missing_locations["dataframe_index"] + 1
    missing_locations = missing_locations[["raw_file_line", "feature"]]
    print("Exact missing-cell locations (raw_file_line is 1-based; line 1 is the first patient record):")
    display(missing_locations)

    missing_row_indices = missing_locations["raw_file_line"] - 1
    rows_with_missing = raw_data.loc[missing_row_indices.unique()].copy()
    rows_with_missing.insert(0, "raw_file_line", rows_with_missing.index + 1)
    print("Patient records containing one or more missing values:")
    display(rows_with_missing.reset_index(drop=True))

# Step 4: drop rows only when the outcome is missing; convert UCI targets 0-4 to binary.
rows_before = len(raw_data)
data = raw_data.dropna(subset=["target_raw"]).copy()
rows_without_target = rows_before - len(data)
data["target"] = (data["target_raw"] > 0).astype(int)
data = data.drop(columns=["target_raw"])

# Step 5: save the model-ready table. Missing predictors are kept for pipeline imputation.
data.to_csv(clean_path, index=False)
print("CLEANED DATA")
print(f"Saved raw data to: {raw_path}")
print(f"Saved cleaned data to: {clean_path}")
print(f"Rows retained: {len(data):,} | Rows removed for missing target: {rows_without_target:,}")
print("Target mapping: 0 = no heart disease; original values 1-4 = heart disease (1).")
clean_missing_summary = pd.DataFrame({
    "missing_count": data.isna().sum(),
    "missing_percent": (data.isna().mean() * 100).round(2),
})
display(clean_missing_summary)
print("First five cleaned rows:")
display(data.head())

RAW DATA - missing-value audit
In the original CSV, a question mark (?) means that a value was not recorded.
Raw rows: 303 | Raw columns: 14


,missing_count,missing_percent
age,0,0.00
sex,0,0.00
cp,0,0.00
trestbps,0,0.00
chol,0,0.00
fbs,0,0.00
restecg,0,0.00
thalach,0,0.00
exang,0,0.00
oldpeak,0,0.00


Exact missing-cell locations (raw_file_line is 1-based; line 1 is the first patient record):


,raw_file_line,feature
0,88,thal
1,167,ca
2,193,ca
3,267,thal
4,288,ca
5,303,ca


Patient records containing one or more missing values:


,raw_file_line,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target_raw
0,88,53.0,0.0,3.0,128.0,216.0,0.0,2.0,115.0,0.0,0.0,1.0,0.0,NaN,0
1,167,52.0,1.0,3.0,138.0,223.0,0.0,0.0,169.0,0.0,0.0,1.0,NaN,3.0,0
2,193,43.0,1.0,4.0,132.0,247.0,1.0,2.0,143.0,1.0,0.1,2.0,NaN,7.0,1
3,267,52.0,1.0,4.0,128.0,204.0,1.0,0.0,156.0,1.0,1.0,2.0,0.0,NaN,2
4,288,58.0,1.0,2.0,125.0,220.0,0.0,0.0,144.0,0.0,0.4,2.0,NaN,7.0,0
5,303,38.0,1.0,3.0,138.0,175.0,0.0,0.0,173.0,0.0,0.0,1.0,NaN,3.0,0


CLEANED DATA
Saved raw data to: d:\03_Semester\Semester03\MLOps\Assignment\heart-disease-mlops\data\raw\heart.csv
Saved cleaned data to: d:\03_Semester\Semester03\MLOps\Assignment\heart-disease-mlops\data\processed\heart_clean.csv
Rows retained: 303 | Rows removed for missing target: 0
Target mapping: 0 = no heart disease; original values 1-4 = heart disease (1).


,missing_count,missing_percent
age,0,0.00
sex,0,0.00
cp,0,0.00
trestbps,0,0.00
chol,0,0.00
fbs,0,0.00
restecg,0,0.00
thalach,0,0.00
exang,0,0.00
oldpeak,0,0.00


First five cleaned rows:


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## How to interpret the missing-value output

In this UCI file, `?` is the missing-data marker. Pandas converts it to `NaN` when the raw file is read. The raw audit identifies every affected feature and its source row, so you do not have to search the CSV manually. For the current Cleveland file, six patient records have missing predictor values: four in `ca` and two in `thal`. They are kept in the cleaned CSV and later imputed by the model pipeline. A row with a missing outcome would instead be excluded because supervised training cannot use an unknown target.